# Render the Dreadnoughts Using the Game Data

In [2]:
from IPython.display import display, HTML
display(HTML("<style>.container { width:70% !important; }</style>"))

### Load the Character Sets

In [3]:
import re
from colors_map import *

def loadCharacterSet(charset_lines, ref_offset=0):
    character_set = {}
    charset_data = []
    char_ref = None
    for l in charset_lines:
        if "CHARACTER" in l:
            if charset_data:
                character_set[char_ref] = charset_data
            charset_data = []
            ref = int(l[61:63],16) + ref_offset
            char_ref = f"{ref:0{2}x}".upper()
             
        m = re.findall(r"[0-1]{8}",l)
        if not m:
            continue
        bits = m[0].strip()
        line_bits = []
        for i in range(0,7,2):
            bitpair = bits[i:i+2]
            line_bits += [bitpair]
            line_bits += [bitpair]
        charset_data += [line_bits]
    character_set[char_ref] = charset_data
    return character_set

charsets_file = "uridium/src/charset.asm"
input_file = open(charsets_file,'r')
charset_lines = input_file.readlines()
charset_lines = charset_lines[1439:]
base_character_set = loadCharacterSet(charset_lines)

character_files = [
    "uridium/src/surface1_charset.asm",
    "uridium/src/surface2_charset.asm",
    "uridium/src/surface3_charset.asm",
]
character_sets = []
for filename in character_files:
    charsets_file = filename
    input_file = open(charsets_file,'r')
    charset_lines = input_file.readlines()
    character_set = base_character_set | loadCharacterSet(charset_lines, ref_offset=0x80)
    character_sets += [character_set]


## Generate PNG images of the Character Sets

In [4]:
from PIL import Image, ImageColor
CHARACTER_COLS = 8
CHARACTER_ROWS = 8

def paintRawCharacter(character_set, character, colors, verticalExpand=False):
    multicol0, multicol1,color,color1 = colors
    
    verticalExpansion = 2 if verticalExpand else 1
    CHARACTER_COLS = 8
    CHARACTER_ROWS = 8 * verticalExpansion

    colormap = {
        "01": multicol0,
        "10": color,
        "11": multicol1,
        "00": color1,
    }
    
    if character not in character_set:
        print(character)
        return
    
    image_width = CHARACTER_COLS
    image_height = CHARACTER_ROWS
    img = Image.new( 'RGBA', (image_width+1, image_height+1))
    pixels = img.load()

    bit_array = character_set[character]
    if verticalExpand:
        expanded_bit_array = []
        for a in bit_array:
            expanded_bit_array += [a,a]
        bit_array = expanded_bit_array
    
    for y, l in enumerate(bit_array):
        for x,bit in enumerate(l):
            if bit == "11":
                continue
            pixel_color = ImageColor.getrgb(c64_to_rgb[colormap[bit]])
            pixels[x,y] = pixel_color
    return img

### Get the Level Specific Color-Schemes and Character Set References

In [5]:
raw_level_color_schemes = """
        .BYTE M_GRAY1,M_GRAY3,M_ORANGE,M_YELLOW,M_ORANGE  ; Level 1
        .BYTE M_BLACK,M_GRAY1,M_LTBLUE,M_LTRED,M_RED      ; Level 2, Level 13
        .BYTE M_BROWN,M_ORANGE,M_ORANGE,M_LTGREEN,M_GREEN ; Level 3, Level 11
        .BYTE M_GRAY1,M_GRAY3,M_ORANGE,M_LTBLUE,M_BLUE    ; Level 4
        .BYTE M_GREEN,M_LTGREEN,M_ORANGE,M_LTBLUE,M_BLUE  ; Level 5
        .BYTE M_ORANGE,M_YELLOW,M_ORANGE,M_GRAY2,M_BLACK  ; Level 6
        .BYTE M_GRAY1,M_CYAN,M_LTGREEN,M_LTRED,M_RED      ; Level 7
        .BYTE M_BLACK,M_GRAY2,M_LTRED,M_LTBLUE,M_BLUE     ; Level 8
        .BYTE M_BLUE,M_LTBLUE,M_ORANGE,M_LTGREEN,M_GREEN  ; Level 9 , Level 14
        .BYTE M_GRAY1,M_GRAY2,M_GRAY2,M_LTRED,M_RED       ; Level 10
        .BYTE M_BROWN,M_ORANGE,M_ORANGE,M_LTGREEN,M_GREEN ; Level 3, Level 11
        .BYTE M_BLUE,M_CYAN,M_ORANGE,M_GRAY2,M_GRAY1      ; Level 12
        .BYTE M_BLACK,M_GRAY1,M_LTBLUE,M_LTRED,M_RED      ; Level 2, Level 13
        .BYTE M_BLUE,M_LTBLUE,M_ORANGE,M_LTGREEN,M_GREEN  ; Level 9 , Level 14
        .BYTE M_RED,M_LTRED,M_ORANGE,M_YELLOW,M_ORANGE    ; Level 15
"""
raw_level_color_schemes = [l[14:57].split(',') for l in raw_level_color_schemes.split('\n')][1:-1]
raw_level_color_schemes = [[x.strip() for x in l] for l in raw_level_color_schemes]
#raw_level_color_schemes = [v for l in raw_level_color_schemes for v in l]
#surface_ram = [temp_surface_ram[v:v+80] for v in range(0, len(temp_surface_ram), 80)]
#surface_ram = ["$"+x.upper() for x in flatten(surface_ram)]
level_colors = [None]
for l in raw_level_color_schemes:
    colors = (color_constants[l[1]], "c64_black", color_constants[l[0]], "c64_white")
    level_colors += [colors]

# The character sets each level uses.
level_charsets = [0,2,0,2,0,1,2,1,0,1,2,0,2,1,0,2]

level_names = [None, "Zinc","Lead","Copper","Silver","Iron","Gold","Platinum","Tungsten","Iridon","Kallisto","Tri-alloy","Quadmium","Ergonite","Galactium","Uridium"]

### Create PNG Images of the Level-Specific Charactersets

In [33]:
level_character_set_images = [None]
for i in range(1,16):
    charset_ref = level_charsets[i]
    colors = level_colors[i]
    character_set = character_sets[charset_ref]
    character_images = {}
    for character_name in character_set:
        img = paintRawCharacter(character_set, character_name, colors)
        character_images["$"+character_name] = img
        if character_name:
            img = img.resize((img.width * 15, img.height * 15), Image.NEAREST)
            img.save(f"surface_characters/{i}_{character_name}.png")
    level_character_set_images += [character_images]

## Read in the Surface Structures Data (surfaceStructureData)

In [39]:
"""
Entries in this array have the structure, e.g.:

.BYTE $01  ; Number of Elements
.BYTE $01  ; Run-Length of element
.BYTE $20  ; Data (List of Character Set references)

"""
game_data_file = "uridium/src/game_data.asm"
lines_in_file = open(game_data_file,'r').readlines()

structure_data_header = lines_in_file[770]
assert ("surfaceStructureData" in structure_data_header)
structure_data_lines = lines_in_file[771:1230]

raw_surface_structure_data = [l[14:].strip() for l in structure_data_lines]
surface_structure_data = [x for l in raw_surface_structure_data for x in l.split(',')]

In [40]:
toInt = lambda x: int(x[1:],16)
surface_structures = []
i = 0
while (i < len(surface_structure_data)):
    num_elements = toInt(surface_structure_data[i])
    i += 1
    structure_data = []
    for j in range(0,num_elements):
        run_length = toInt(surface_structure_data[i])
        i += 1
        run_length = run_length & 0x1F
        raw_data = surface_structure_data[i:i+run_length]
        i += run_length
        data = [x[1:] for x in raw_data]
        structure_data += [data]
    surface_structures += [structure_data]

#for i,s in enumerate(surface_structures):
#    print(hex(i),s)

### Generate PNG Images of all the Structures

In [65]:
surface_strip_images = [None]
for j in range(1,16):
    character_images = level_character_set_images[j]
    strip_images = {}
    for i,surface_structure in enumerate(surface_structures):
        if not surface_structure:
            continue
        strip_max = max([len(x) for x in surface_structure])
        HEIGHT = 8 * strip_max
        WIDTH = len(surface_structure) * 8
        x,y = 0,HEIGHT-8
        img = Image.new( 'RGBA', (WIDTH,HEIGHT))
        for strip in surface_structure:
            for c in strip:
                char_image = character_images["$"+c]
                img.paste(char_image,(x,y),mask=char_image)
                y-=8
            x+=8
            y=HEIGHT-8
        ref_offset = i+1
        ref = ("00" + hex(ref_offset)[2:].upper())[-2:]
        strip_images[ref] = img

        img = img.resize((img.width * 5, img.height * 5), Image.NEAREST)
        img.save(f"surface_structures/{j}_{i}.png")
        img.save(f"surface_structures_hex/{j}_{ref}.png")
    surface_strip_images += [strip_images]

### Read in the Dreadnought Data for All Levels

In [84]:
def getDreadnoughtData(level_data_lines):
    raw_dreadnought_data = [l[14:].strip() for l in level_data_lines]
    dreadnought_data = [x for l in raw_dreadnought_data for x in l.split(',')]

    end_of_base_data = dreadnought_data.index('$00')
    base_data = [x[1:] for x in dreadnought_data[:end_of_base_data]]
    raw_location_data = dreadnought_data[end_of_base_data+1:]
    location_data = []
    for i in range(0, len(raw_location_data), 3):
        hi_byte = int(raw_location_data[i][1:],16)
        if not hi_byte:
            break
        hi_byte |= 0x80
        hi_byte &= 0xBF
        hi_byte = hex(hi_byte)[2:]
        lo_byte = raw_location_data[i+1][1:]
        hex_location = hi_byte + lo_byte
        location = int(hex_location,16)
        structure = raw_location_data[i+2][1:]
        location_data += [(hex_location, location,structure)]
    dreadnought_surface = (base_data,location_data)
    return dreadnought_surface

# Read in bulk of the dreadnought data from level data first.
level_data_file = "uridium/src/level_data.asm"
lines_in_file = open(level_data_file,'r').readlines()

dreadnought_surfaces = {}
level_data_lines = []
current_dreadnought = ""
for l in lines_in_file:
    if l.startswith("level") and "DreadnoughtData" in l:
        if current_dreadnought:
            dreadnought_surfaces[current_dreadnought] = getDreadnoughtData(level_data_lines)
        current_dreadnought = l[:22].strip()
        level_data_lines = []
    if ".BYTE" not in l:
        continue
    level_data_lines += [l]
dreadnought_surfaces[current_dreadnought] = getDreadnoughtData(level_data_lines)

# Read in remaining dreadnoughts from charset.asm
level_data_file = "uridium/src/charset.asm"
lines_in_file = open(level_data_file,'r').readlines()

assert "level7DreadnoughtData" in lines_in_file[1297]
level_data_lines = [l for l in lines_in_file[1298:1358] if ".BYTE" in l]
dreadnought_surfaces["level7DreadnoughtData"] = getDreadnoughtData(level_data_lines)

assert "level8DreadnoughtData" in lines_in_file[1358]
level_data_lines = [l for l in lines_in_file[1359:1425] if ".BYTE" in l]
dreadnought_surfaces["level8DreadnoughtData"] = getDreadnoughtData(level_data_lines)


### Render the Dreadnought for All Levels

In [183]:
import math

LEVEL = 8

LOCATION_START = 0x8200
LOCATION_END = 0xA400
DREADNOUGHT_COLS = 0x200

width = DREADNOUGHT_COLS * 8 # 4096
height = int((LOCATION_END - LOCATION_START) / DREADNOUGHT_COLS) * 8 # 128

level_images = [None]
for LEVEL in range(1,16):
    img = Image.new( 'RGBA', (width,height))

    base_data, location_data = dreadnought_surfaces[f"level{LEVEL}DreadnoughtData"]
    strip_images = surface_strip_images[LEVEL]

    x,y = 512,0
    for structure in base_data:
        strip_image = strip_images[structure]
        # Y position is relative to the bottom of the map.
        y = height - strip_image.height
        img.paste(strip_image,(x,y),mask=strip_image)
        x+= strip_image.width

    for hex_offset, location, structure in location_data:
        if structure == "20":
            print(structure)
        if structure not in strip_images:
            continue
        strip_image = strip_images[structure]
        offset = location - LOCATION_START
        x = (offset % DREADNOUGHT_COLS) * 8
        # The location given is for the bottom of the object (uridium draws upwards)
        y = (int(math.ceil(offset / DREADNOUGHT_COLS)) * 8) - strip_image.height
        img.paste(strip_image,(x,y),mask=strip_image)
    if LEVEL == 12:
        img = img.crop((350,0,img.width-470,img.height))
    elif LEVEL == 15:
        img = img.crop((410,0,img.width-200,img.height))
    elif LEVEL == 9:
        img = img.crop((480,0,img.width-300,img.height))
    elif LEVEL == 2:
        img = img.crop((480,0,img.width-400,img.height))
    elif LEVEL == 10:
        img = img.crop((480,0,img.width-430,img.height))
    elif LEVEL == 13:
        img = img.crop((480,0,img.width-430,img.height))
    else:
        img = img.crop((480,0,img.width-450,img.height))
    #img = img.resize((img.width * 5, img.height * 5), Image.NEAREST)
    img.save(f"surfaces_raw/{LEVEL}.png")
    level_images += [img]

### Now make diagrams of all the levels

In [236]:
from PIL import Image, ImageDraw, ImageColor, ImageFont

def generateSurfaceDiagram(level_name, level_number,level_image):
    level_image = level_image.rotate(90,expand=1)
    level_image = level_image.resize((level_image.width * 2, level_image.height * 2), Image.NEAREST)
    
    img = Image.new('RGBA', (70 + level_image.width + 10, level_image.height + 5))
    draw = ImageDraw.Draw(img)
    draw.rectangle([(0,0),img.size], fill = "white")

    # Sprite label
    label_text = f"URIDIUM/{('00'+str(level_number))[-2:]}/{level_name}"
    label_fnt_size = 64
    label_fnt = ImageFont.truetype("Eurostile.ttf", label_fnt_size)
    txt_width = len(label_text) * label_fnt_size + 10
    txt = Image.new('RGBA', (txt_width, label_fnt_size))
    draw = ImageDraw.Draw(txt)
    draw.rectangle([(0,0), txt.size], fill = "white")
    draw.text((0, 0), label_text, font=label_fnt, fill="black")
    txt = txt.rotate(90, expand=1)
    img.paste(txt, (5, img.height - txt.height - 70))

    # Main Sprite image
    img.paste(level_image, (70,10),mask=level_image)

    return img

def generateSurfaceDiagramHorizontal(level_name, level_number,level_image):
    level_image = level_image.resize((level_image.width * 2, level_image.height * 2), Image.NEAREST)
    
    img = Image.new('RGBA', (level_image.width, 70 + level_image.height))
    draw = ImageDraw.Draw(img)
    draw.rectangle([(0,0),img.size], fill = "white")

    # Sprite label
    label_text = f"URIDIUM/{('00'+str(level_number))[-2:]}/{level_name}"
    label_fnt_size = 64
    label_fnt = ImageFont.truetype("Eurostile.ttf", label_fnt_size)
    txt_width = len(label_text) * label_fnt_size + 10
    txt = Image.new('RGBA', (txt_width, label_fnt_size))
    draw = ImageDraw.Draw(txt)
    draw.rectangle([(0,0), txt.size], fill = "white")
    draw.text((0, 0), label_text, font=label_fnt, fill="black")
    img.paste(txt, (10, 5))

    # Main Sprite image
    img.paste(level_image, (-50,70),mask=level_image)

    return img


In [214]:
for i, n in enumerate(level_names):
    if not n:
        continue
    img = generateSurfaceDiagram(n.upper(), i, level_images[i])
    img.save(f"surface_diagrams/{i}_{n}.png")
    img.save(f"surface_diagrams/{i}.png")

In [239]:
for i, n in enumerate(level_names):
    if not n:
        continue
    img = generateSurfaceDiagramHorizontal(n.upper(), i, level_images[i])
    img.save(f"surface_diagrams/{i}_{n}_horizontal.png")
    img.save(f"surface_diagrams/{i}_horizontal.png")

    #half_width = int(img.width / 2)
    half_width = 3166
    img1 = img.crop((0,0,half_width,img.height))
    img2 = img.crop((half_width,0,img.width,img.height))
    img1.save(f"surface_diagrams/{i}_horizontal_1.png")
    img2.save(f"surface_diagrams/{i}_horizontal_2.png")
    

# Scratchpad